In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym
import numpy as np

# تنظیمات دستگاه
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# شبکه Actor-Critic
class ActorCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ActorCritic, self).__init__()
        # Shared backbone
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        # Actor head (سیاست)
        self.actor = nn.Linear(hidden_dim, output_dim)
        # Critic head (ارزش وضعیت)
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        shared_out = self.shared(x)
        # خروجی احتمال عمل‌ها (لگیت‌ها)
        logits = self.actor(shared_out)
        # توزیع کتگوریکال برای انتخاب عمل
        probs = torch.softmax(logits, dim=-1)
        # خروجی ارزش وضعیت
        value = self.critic(shared_out)
        return probs, value

# محیط
env = gym.make('CartPole-v1')

# پارامترها
input_dim = env.observation_space.shape[0]  # 4
output_dim = env.action_space.n             # 2
hidden_dim = 128
lr = 0.001
gamma = 0.99
batch_size = 32  # تعداد episode در هر batch
num_batches = 200

# شبکه Actor-Critic
model = ActorCritic(input_dim, hidden_dim, output_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

# تابع محاسبه بازده با تخفیف
def compute_returns(rewards, gamma):
    R = 0
    returns = []
    for r in reversed(rewards):
        R = r + gamma * R
        returns.insert(0, R)
    return returns

# آموزش Batch Actor-Critic
for batch in range(num_batches):
    log_probs = []
    values = []
    rewards = []
    masks = []
    total_rewards = []

    # جمع‌آوری داده از چندین episode (یک batch)
    for _ in range(batch_size):
        state = env.reset()[0]
        episode_log_probs = []
        episode_values = []
        episode_rewards = []
        episode_masks = []

        done = False
        while not done:
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            probs, value = model(state_tensor)

            # نمونه‌گیری عمل
            action_dist = torch.distributions.Categorical(probs)
            action = action_dist.sample()
            log_prob = action_dist.log_prob(action)

            next_state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated

            episode_log_probs.append(log_prob)
            episode_values.append(value.squeeze())
            episode_rewards.append(reward)
            episode_masks.append(0 if done else 1)

            state = next_state

        # ذخیره داده‌های episode
        log_probs.append(episode_log_probs)
        values.append(episode_values)
        rewards.append(episode_rewards)
        masks.append(episode_masks)
        total_rewards.append(sum(episode_rewards))

    # محاسبه بازده برای تمام episodeها
    all_returns = []
    all_log_probs = []
    all_values = []

    for i in range(batch_size):
        returns = compute_returns(rewards[i], gamma)
        all_returns.extend(returns)
        all_log_probs.extend(log_probs[i])
        all_values.extend(values[i])

    # تبدیل به تانسور
    returns = torch.tensor(all_returns).float().to(device)
    values = torch.stack(all_values)
    log_probs = torch.stack(all_log_probs)

    # محاسبه Advantage: A(s,a) = G - V(s)
    advantages = returns - values.squeeze()

    # نرمال‌سازی Advantage (اختیاری، اما پایداری را افزایش می‌دهد)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # تابع زیان: Actor loss + Critic loss
    actor_loss = -(log_probs * advantages.detach()).mean()  # فقط گرادیان از log_prob بگیرد
    critic_loss = advantages.pow(2).mean()  # MSE: (G - V)^2
    total_loss = actor_loss + 0.5 * critic_loss

    # به‌روزرسانی وزن‌ها
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    # نمایش میانگین بازده در این بچ
    avg_reward = np.mean(total_rewards)
    if batch % 20 == 0:
        print(f"Batch {batch}, Average Reward: {avg_reward:.2f}, Loss: {total_loss.item():.4f}")

print("Training finished.")